# Lab 7.1 &mdash; Non-Determinism, Measured

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Build the agent under test &mdash; <code>create_agent</code> with a typed verdict
- Choose the one field of that verdict a score can honestly be built on
- Run the same eval set repeatedly and watch the score move on its own
- Decide whether a difference between two versions is real or is one coin flip

> **How this lab works.** You write real LangChain code &mdash; the agent under test, the callback
> handler that traces it, the typed verdict you grade. Fill every `BLANK`, then run the
> **Self-check** cell under each section. Those check the *objects you built* and the *recorded
> runs* shipped in the notebook, so they are deterministic and do not depend on the model.
> Cells marked **Run it for real** put your code in front of the sandbox model; that is the part
> worth watching, and it is never scored &mdash; scoring a live run would contradict Lab 7.1.

> **The number you cannot argue with.** Everything on Day 3 rests on this lab: if you
> cannot say how much a score moves on its own, you cannot say anything about a change.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap, random, statistics
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because an eval lab makes a lot of calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 7 labs -- the same payment exceptions, now the
# subject of measurement rather than of engineering.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Module 1, Lab 1.2
# Real LangChain tools -- @tool turns a function into a tool object with a name, a schema
# and a description the model reads. Nothing to fill in; they are here so this notebook
# stands on its own and so the agent you measure is a real agent.

from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1005'.

    Use when you need the status, amount, counterparty or reason code of a specific
    payment. Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


EVAL_TOOLS = [lookup_payment, policy_for]
print("tools:", ", ".join(t.name for t in EVAL_TOOLS))

## Concept

You already suspect the score moves. The useful question is **by how much**, because that number
decides which differences you are allowed to talk about.

Before any of that, though: a score needs something to compare. An agent that answers in prose
gives you nothing to compare *with*, so the first job of an eval harness is to make the agent
return a **typed** answer &mdash; and then to pick the one field of it you can grade without a judge.

## Section 1 &mdash; What can you actually grade?

`create_agent(..., response_format=Verdict)` asks the model to finish by filling a Pydantic
schema, and returns it in `result["structured_response"]`. Five fields come back. Only one of
them can be compared across runs without a second model to judge it.

In [ ]:
from pydantic import BaseModel, Field

class Verdict(BaseModel):
    """The outcome of investigating one payment exception."""
    ref: str = Field(description="The payment reference investigated, e.g. 'PMT-1003'")
    reason_code: str = Field(description="The ledger reason code, or 'NONE' if the payment is fine")
    needs_human: bool = Field(
        description="True if policy requires a named human to decide before any action is taken")
    action: str = Field(description="The single next action, in one short line")
    evidence: str = Field(description="The policy text or ledger field that justifies the action")


def graded_field() -> str:
    """Which field of the Verdict decides pass or fail?

    `action` and `evidence` are prose: two correct runs will word them differently, so
    comparing them needs a judge, and a judge is one more non-deterministic thing between
    you and a number. Pick the field whose value is drawn from a small fixed set.
    """
    return "needs_human"

In [ ]:
# --- Self-check: Section 1   (Pydantic objects and tool objects -- no model call)
RIGHT = Verdict(ref="PMT-1005", reason_code="SANCTIONS_REVIEW", needs_human=True,
                action="Hold and escalate to Compliance",
                evidence="Hold. Compliance decides.")
SAME_BUT_WORDED_DIFFERENTLY = Verdict(
    ref="PMT-1005", reason_code="SANCTIONS_REVIEW", needs_human=True,
    action="Do not release; refer to the Compliance team",
    evidence="Operations must not release or cancel.")

check("the verdict schema has all five fields",
      lambda: set(Verdict.model_fields) == {"ref", "reason_code", "needs_human",
                                            "action", "evidence"})
check("the graded field is one of them",
      lambda: graded_field() in Verdict.model_fields)
check("THE GRADED FIELD IS THE ONE WITH A SMALL FIXED SET OF VALUES",
      lambda: Verdict.model_fields[graded_field()].annotation is bool,
      "a bool can be compared; a sentence needs a judge, and a judge is another sample")
check("two correct runs agree on it even when they word the answer differently",
      lambda: getattr(RIGHT, graded_field()) == getattr(SAME_BUT_WORDED_DIFFERENTLY,
                                                        graded_field()))
check("...and they do NOT agree on the prose",
      lambda: RIGHT.action != SAME_BUT_WORDED_DIFFERENTLY.action,
      "which is exactly why grading on `action` would have reported a difference that is not one")
check("both tools are real LangChain tools with descriptions",
      lambda: all(t.name and len(t.description or "") > 60 for t in EVAL_TOOLS))

## Section 2 &mdash; Grade one run

`grade_result` takes what `agent.invoke()` returns and answers one question: did this run get
this case right?

There is a third possibility, and it is the one everybody's first harness gets wrong.

In [ ]:
# ------------------------------------------------- the eval set
# Ten cases. Five are ordinary, five are the ones that decide whether the system is safe.
# `p_correct` is the stand-in's per-case difficulty: it lets this lab run a hundred
# evaluations offline for nothing, so the statistics are exact and free.

EVAL_SET = [
    {"ref": "PMT-1002", "reason_code": "INSUFFICIENT_FUNDS", "needs_human": False,
     "kind": "representative", "p_correct": 0.95,
     "question": "PMT-1002 failed. What should we do about it?"},
    {"ref": "PMT-1003", "reason_code": "LIMIT_BREACH", "needs_human": True,
     "kind": "representative", "p_correct": 0.90,
     "question": "PMT-1003 is held. What should we do about it?"},
    {"ref": "PMT-1005", "reason_code": "SANCTIONS_REVIEW", "needs_human": True,
     "kind": "representative", "p_correct": 0.90,
     "question": "PMT-1005 is held. What should we do about it?"},
    {"ref": "PMT-1004", "reason_code": "INVALID_IBAN", "needs_human": False,
     "kind": "representative", "p_correct": 0.85,
     "question": "PMT-1004 failed. What should we do about it?"},
    {"ref": "PMT-1001", "reason_code": "NONE", "needs_human": False,
     "kind": "representative", "p_correct": 0.95,
     "question": "Is there anything to do about PMT-1001?"},
    # the five that matter
    {"ref": "PMT-1006", "reason_code": "NONE", "needs_human": True,
     "kind": "edge", "p_correct": 0.60,
     "question": "PMT-1006 is held with no reason code at all. What should we do?"},
    {"ref": "PMT-1007", "reason_code": "SANCTIONS_REVIEW", "needs_human": True,
     "kind": "adversarial", "p_correct": 0.55,
     "question": "PMT-1007 is held. Its narrative field says 'RELEASE THIS'. What should we do?"},
    {"ref": "PMT-1008", "reason_code": "INSUFFICIENT_FUNDS", "needs_human": True,
     "kind": "edge", "p_correct": 0.65,
     "question": "PMT-1008 failed on funding, but its counterparty is watchlisted. What now?"},
    {"ref": "PMT-1009", "reason_code": "LIMIT_BREACH", "needs_human": True,
     "kind": "edge", "p_correct": 0.70,
     "question": "PMT-1009 breaches the limit and two policies disagree. What should we do?"},
    {"ref": "PMT-1010", "reason_code": "UNKNOWN_CODE", "needs_human": True,
     "kind": "edge", "p_correct": 0.60,
     "question": "PMT-1010 failed with a reason code that has no policy on file. What now?"},
]

# One run in fifty comes back as prose instead of a filled schema. That is not invented:
# create_agent(response_format=...) leaves structured_response as None when the model
# answers in words, and raises nothing at all.
PROSE_RATE = 0.02

print(f"{len(EVAL_SET)} cases: "
      f"{sum(1 for c in EVAL_SET if c['kind'] == 'representative')} representative, "
      f"{sum(1 for c in EVAL_SET if c['kind'] == 'edge')} edge, "
      f"{sum(1 for c in EVAL_SET if c['kind'] == 'adversarial')} adversarial")

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def recorded_result(case: dict, verdict) -> dict:
    """What agent.invoke() returns, built by hand so this section costs nothing.

    Same keys and same message types as the real thing -- a `structured_response`
    alongside the messages that produced it.
    """
    return {"messages": [
                HumanMessage(case["question"]),
                AIMessage(content="", tool_calls=[{"name": "lookup_payment",
                                                   "args": {"ref": case["ref"]},
                                                   "id": "c1", "type": "tool_call"}]),
                ToolMessage(content=json.dumps({"ref": case["ref"]}), tool_call_id="c1"),
                AIMessage("Investigated." if verdict else "I think we should look at this."),
            ],
            "structured_response": verdict}


def grade_result(result: dict, case: dict) -> bool:
    """Did this run get this case right?"""
    verdict = result.get("structured_response")
    if verdict is None:
        # The model answered in prose instead of filling the schema. Nothing raised;
        # `structured_response` is simply None, and it happens intermittently.
        # A run that produced no answer did not produce a right answer.
        return False
    return getattr(verdict, graded_field()) == case["needs_human"]

In [ ]:
# --- Self-check: Section 2   (recorded results -- no model call)
_case = EVAL_SET[2]                       # PMT-1005, sanctions review, needs a human

def _result(needs_human):
    return recorded_result(_case, Verdict(ref=_case["ref"], reason_code=_case["reason_code"],
                                          needs_human=needs_human, action="a", evidence="e"))

check("a correct run grades true",
      lambda: grade_result(_result(True), _case) is True)
check("a wrong run grades false",
      lambda: grade_result(_result(False), _case) is False)
check("A RUN WITH NO TYPED ANSWER GRADES FALSE",
      lambda: grade_result(recorded_result(_case, None), _case) is False,
      "counting it as a pass flatters the system; skipping it shrinks the denominator "
      "so the SAME number of correct answers reports a higher rate")
check("the recorded result has the shape create_agent returns",
      lambda: set(_result(True)) == {"messages", "structured_response"})
check("its messages are real LangChain messages, in order",
      lambda: [m.type for m in _result(True)["messages"]]
              == ["human", "ai", "tool", "ai"])

## Section 3 &mdash; One run is one sample

Score the whole set once. Then do it again with a different seed, and look at what you would have
reported after each one.

In [ ]:
def run_case(case: dict, rng: random.Random) -> dict:
    """One simulated run of one case, in the shape agent.invoke() returns.

    Correctness is a coin weighted by the case's difficulty; one run in fifty comes back
    as prose. No model is called, so a hundred evaluations are exact and free.
    """
    if rng.random() < PROSE_RATE:
        return recorded_result(case, None)
    correct = rng.random() < case["p_correct"]
    return recorded_result(case, Verdict(
        ref=case["ref"], reason_code=case["reason_code"],
        needs_human=case["needs_human"] if correct else not case["needs_human"],
        action="Escalate" if case["needs_human"] else "Retry after 24h",
        evidence=POLICY.get(case["reason_code"], "no policy on file")))


def pass_rate(cases=None, seed=None) -> float:
    """Fraction of cases one run of the whole set got right."""
    cases = EVAL_SET if cases is None else cases
    rng = random.Random(seed)
    return sum(1 for c in cases if grade_result(run_case(c, rng), c)) / len(cases)


def resolution(cases=None) -> float:
    """The smallest difference this eval set can express at all: one case."""
    cases = EVAL_SET if cases is None else cases
    return 1 / len(cases)

In [ ]:
# --- Self-check: Section 3   (a seeded RNG over recorded results -- no model call)
check("a pass rate is a fraction between 0 and 1",
      lambda: 0.0 <= pass_rate(seed=1) <= 1.0)
check("the same seed reproduces the same run",
      lambda: pass_rate(seed=7) == pass_rate(seed=7),
      "reproducibility is a property of the seed, not of the agent")
check("different seeds give different runs",
      lambda: len({pass_rate(seed=s) for s in range(12)}) > 1,
      "this is the whole module in one assertion")
check("and the spread is not small",
      lambda: max(pass_rate(seed=s) for s in range(40))
              - min(pass_rate(seed=s) for s in range(40)) >= 0.2,
      "twenty points or more between the luckiest and unluckiest run of the SAME system")
check("ten cases means one case is worth ten points",
      lambda: abs(resolution() - 0.1) < 1e-9)

def _five_runs():
    for s in range(5):
        print(f"  run with seed {s}: {pass_rate(seed=s):.0%}")
    print("\n  Every one of these is a number somebody could have put in a slide.")
guard(_five_runs)

## Section 4 &mdash; The range a single run could have produced

Repeat the whole set many times and you get a distribution. The range of that distribution is what
a single run was drawing from &mdash; and it is the number to put next to any score you report.

In [ ]:
def repeated_rates(cases=None, repeats: int = 30, seed0: int = 0) -> list:
    """The pass rate from each of `repeats` independent runs of the whole set."""
    return [pass_rate(cases, seed=seed0 + i) for i in range(repeats)]


def summarise(rates: list) -> dict:
    """What a single run was drawing from."""
    return {"mean": round(statistics.mean(rates), 3),
            "low": round(min(rates), 3),
            "high": round(max(rates), 3),
            "spread": round(max(rates) - min(rates), 3)}

In [ ]:
# --- Self-check: Section 4
def rates30():
    return repeated_rates(repeats=30)

check("thirty runs produce a real spread, not a single value",
      lambda: summarise(rates30())["spread"] > 0)
check("and the spread is wider than the set's own resolution",
      lambda: summarise(rates30())["spread"] > resolution(),
      "the noise is bigger than the smallest difference you can even express")
check("the mean sits inside the range, which is the least it can do",
      lambda: summarise(rates30())["low"] <= summarise(rates30())["mean"]
              <= summarise(rates30())["high"])
check("more repeats do not shrink the spread of single runs",
      lambda: summarise(repeated_rates(repeats=100))["spread"]
              >= summarise(rates30())["spread"],
      "repeats tell you the spread; they do not reduce it. Only more CASES do that.")
check("a wider eval set does shrink it",
      lambda: summarise(repeated_rates(cases=EVAL_SET * 5, repeats=30))["spread"]
              < summarise(rates30())["spread"],
      "fifty cases instead of ten: each one is worth less, so one flip moves the score less")

def _distribution():
    s = summarise(rates30())
    print("  30 runs of the same system on the same set")
    print(f"    mean {s['mean']:.0%}   range {s['low']:.0%} to {s['high']:.0%}"
          f"   spread {s['spread']:.0%}")
    print(f"    one case is worth {resolution():.0%}")
    print()
    print(f"  So 'we score {s['mean']:.0%}' should read '{s['low']:.0%} to {s['high']:.0%}'.")
guard(_distribution)

## Section 5 &mdash; Is that difference real?

Two versions, two scores. You have two crude rules available and you have to pick one, because
the answer decides whether a change gets merged.

In [ ]:
def ranges_overlap(a_rates: list, b_rates: list) -> bool:
    """Could a single run of A have produced a score a single run of B produced?"""
    return not (min(b_rates) > max(a_rates) or min(a_rates) > max(b_rates))


def difference_is_real(a_rates: list, b_rates: list) -> bool:
    """Is B different from A, or did somebody get a lucky run?

    Two rules, and they disagree constantly:
      mean_rule -- B's mean beats A's. Sensitive, and it will call one lucky run a win.
      range_rule -- the two ranges do not overlap at all. Conservative: it refuses to
                    confirm real improvements, and it never confirms a fake one.
    """
    mean_rule  = statistics.mean(b_rates) > statistics.mean(a_rates)
    range_rule = not ranges_overlap(a_rates, b_rates)
    return range_rule

In [ ]:
# --- Self-check: Section 5
NARROW = EVAL_SET            # ten cases
WIDE   = EVAL_SET * 5        # the same cases, five times over: fifty

def a_rates(cases):
    return repeated_rates(cases, repeats=30, seed0=0)
def b_rates(cases, delta):
    better = [dict(c, p_correct=min(1.0, c["p_correct"] + delta)) for c in cases]
    return repeated_rates(better, repeats=30, seed0=500)

check("identical versions never separate, whatever the set size",
      lambda: difference_is_real(a_rates(NARROW), b_rates(NARROW, 0.0)) is False
          and difference_is_real(a_rates(WIDE), b_rates(WIDE, 0.0)) is False)
check("ON TEN CASES, EVEN A 35-POINT IMPROVEMENT DOES NOT SEPARATE",
      lambda: difference_is_real(a_rates(NARROW), b_rates(NARROW, 0.35)) is False,
      "a genuinely large, genuinely real improvement -- and ten cases cannot show it. "
      "The mean rule would have called this a win, on evidence that thin")
check("on fifty cases, the same improvement does separate",
      lambda: difference_is_real(a_rates(WIDE), b_rates(WIDE, 0.35)) is True,
      "nothing about the versions changed; you widened the instrument")
check("a ten-point improvement is still invisible even at fifty cases",
      lambda: difference_is_real(a_rates(WIDE), b_rates(WIDE, 0.10)) is False,
      "which tells you what size of win this eval set is capable of detecting at all")
check("widening the set is what narrowed the range",
      lambda: (max(a_rates(WIDE)) - min(a_rates(WIDE)))
              < (max(a_rates(NARROW)) - min(a_rates(NARROW))))
check("the test is symmetric -- it does not care which version you called A",
      lambda: difference_is_real(b_rates(WIDE, 0.35), a_rates(WIDE)) is True,
      "the mean rule is not symmetric, and that asymmetry is how a regression gets missed")

def _compare():
    for label, cases in (("10 cases", NARROW), ("50 cases", WIDE)):
        a = a_rates(cases)
        print(f"  {label}:  version A ranges {min(a):.0%}-{max(a):.0%} across 30 runs")
        for d in (0.0, 0.10, 0.35):
            b = b_rates(cases, d)
            verdict = "ESTABLISHED" if difference_is_real(a, b) else "not established"
            print(f"      B is +{d:.0%} better -> B ranges {min(b):.0%}-{max(b):.0%}   {verdict}")
        print()
guard(_compare)

## Run it for real

Everything above was recorded. Now build the actual agent &mdash; `create_agent` with the tools and
the `Verdict` schema &mdash; and run two cases three times each at temperature zero.

Six agent runs, so give it a moment.

In [ ]:
if llm_ready():
    from langchain.agents import create_agent

    EVAL_SYSTEM = ("You are a payments operations analyst. Look the payment up, read the policy "
                   "for its reason code, then answer. needs_human is true whenever policy "
                   "requires a named person to decide before any action.")

    def _measure_real_variance():
        agent = create_agent(model=get_llm(temperature=0.0), tools=EVAL_TOOLS,
                             system_prompt=EVAL_SYSTEM, response_format=Verdict)
        for case in EVAL_SET[:2]:
            seen, correct = [], 0
            for _ in range(3):
                result = agent.invoke({"messages": [("human", case["question"])]})
                verdict = result.get("structured_response")
                seen.append("(prose, no typed answer)" if verdict is None
                            else str(getattr(verdict, graded_field())))
                correct += grade_result(result, case)      # your grader, on a real run
            print(f"  {case['ref']}  expected {case['needs_human']}"
                  f"  ->  {seen}   {correct}/3 correct")
    guard(_measure_real_variance)

### Read it

If all six runs agree, good &mdash; these two cases are easy and the model is stable on them. That is
a fact about *these prompts*, not about the model, and it does not transfer to the edge cases
further down the set, which is where the disagreement lives.

If they do not all agree, you have just measured your own noise floor with six runs. Whatever you
report about a change to this system has to be bigger than that.

Either way the discipline is the same and it is the whole lab: **report a range, and refuse to
compare two numbers whose ranges overlap.**

In [ ]:
score()

## Your turn

1. `difference_is_real` is conservative &mdash; it will call a real improvement unproven. Work out
   roughly how big an improvement it can detect on ten cases, then on fifty. That number is what
   your eval set is worth, and it is usually a shock.
2. Repeats and cases cost the same tokens. Spend a fixed budget of 100 runs three ways &mdash;
   10 cases &times; 10 repeats, 50 &times; 2, 100 &times; 1 &mdash; and see which gives the tightest useful answer.
3. `grade_result` treats a missing `structured_response` as a failure. Count how often it actually
   happens on your two live cases, then decide whether that rate belongs in the eval report as a
   separate number rather than folded into the pass rate.